In [ ]:
import pandas as pd
import requests
import json
import time
import datetime
import uuid

In [ ]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}

In [ ]:
df_source = pd.read_json("best_cities_france.json")
df_source = df_source.rename(columns={0: "city"})
df_source.head()

In [ ]:
df = df_source.copy()
for index, row in df.iterrows():
    print(index, row["city"])
    res = requests.get(f"https://nominatim.openstreetmap.org/search?q={row['city']},France&format=json", headers=headers)
    city = res.json()[0]
    df.loc[index, "lat"] = city["lat"]
    df.loc[index, "lon"] = city["lon"]
    time.sleep(1)
df.head()
df.to_csv("cities_with_geoposition.csv", index=False)

In [ ]:
df = pd.read_csv("cities_with_geoposition.csv")
df.head()

In [ ]:
list_weather_data = []

for index, row in df.iterrows():
    res_weather = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={row['lat']}&lon={row['lon']}&units=metric&exclude=current,minutely,hourly,alerts&appid=4656ed7337e689999007412af6a4dafe", headers=headers)
    res_weather_json = res_weather.json()
    
    for res in res_weather_json['list']:
        weather_entry = {
            "city": row['city'],
            "lat": row['lat'],
            "lon": row['lon'],
            "date": datetime.datetime.fromtimestamp(res['dt']).strftime('%d/%m/%Y'),
            "hour": datetime.datetime.fromtimestamp(res['dt']).strftime('%H:%M'),
            "temp": res['main']['temp'],
            "prob_rain": res['pop'],
            "volume_rain": res.get('rain', {}).get('3h', 0),
            "wind_speed": res['wind']['speed'],
            "perc_cloud": res['clouds']['all']
        }
        list_weather_data.append(weather_entry)
    time.sleep(1)

df_weather = pd.DataFrame(list_weather_data)
print(df_weather.head())
df_weather.to_csv("weather_forecast.csv", index=False)

In [ ]:
df_weather = pd.read_csv("weather_forecast.csv")
df_weather.head()

In [ ]:
df_weather['prob_rain'] = df_weather['prob_rain'] * 100
df_weather['wind_speed'] = df_weather['wind_speed'] * 3.6
df_weather.head()

In [ ]:
df_weather_groupby = df_weather.groupby(['city', 'lat', 'lon', 'date']).agg({'temp': ['mean', 'min', 'max'], 'prob_rain': 'max', 'volume_rain': {'mean', 'max', 'sum'}, 'wind_speed': 'max', 'perc_cloud': 'mean'}).reset_index()
df_weather_groupby.head()

In [ ]:
# --- DEFINITION DU SCORE METEO ---

# 1. Création des scores unitaires (Normalisation sur 100)
# Objectif : 100 = Parfait, 0 = Horrible

# Température (Cible 25°C) : On perd 4 pts par degré d'écart
# .clip(lower=0) empêche d'avoir des notes négatives
df_weather_groupby['score_temp'] = 100 - (abs(df_weather_groupby[('temp', 'max')] - 25) * 4)
df_weather_groupby['score_temp'] = df_weather_groupby['score_temp'].clip(lower=0)

# Pluie Probabilité : 0% = 100 pts. 100% = 0 pts.
# (Note : prob_rain est entre 0 et 1, donc x100 pour le mettre en %)
df_weather_groupby['score_rain_prob'] = 100 - df_weather_groupby[('prob_rain', 'max')]

# Pluie Volume : 0mm = 100 pts. On perd 5 pts par mm.
df_weather_groupby['score_rain_vol'] = 100 - (df_weather_groupby[('volume_rain', 'sum')] * 5)
df_weather_groupby['score_rain_vol'] = df_weather_groupby['score_rain_vol'].clip(lower=0)

# Vent (km/h) : 0 km/h = 100 pts. On perd 1 pt par km/h.
# Attention : wind_speed est en m/s, donc on convertit en km/h (* 3.6)
df_weather_groupby['score_wind'] = 100 - (df_weather_groupby[('wind_speed', 'max')] * 3.6)
df_weather_groupby['score_wind'] = df_weather_groupby['score_wind'].clip(lower=0)

# Nuages : 0% = 100 pts.
df_weather_groupby['score_cloud'] = 100 - df_weather_groupby[('perc_cloud', 'mean')]

# 2. Score Final Pondéré
# Poids : Temp(30%), PluieProba(20%), PluieVol(30%), Vent(10%), Nuages(10%)
df_weather_groupby['total_score'] = (
    df_weather_groupby['score_temp'] * 0.3 +
    df_weather_groupby['score_rain_prob'] * 0.2 +
    df_weather_groupby['score_rain_vol'] * 0.3 +
    df_weather_groupby['score_wind'] * 0.1 +
    df_weather_groupby['score_cloud'] * 0.1
)

# 3. Afficher le TOP 5 des villes (Moyenne sur la période)
top_cities = df_weather_groupby.groupby(['city', 'lat', 'lon'])['total_score'].mean().sort_values(ascending=False)
print("--- FINAL RANKING ---")
print(top_cities.head(10))

# Afficher les détails
df_weather_groupby.head()

In [ ]:
df_weather_groupby.to_csv("weather_forecast_with_score.csv", index=False)

In [ ]:
df_top_cities = pd.DataFrame(top_cities.reset_index())
df_top_cities_top_10 = df_top_cities.loc[0:9,:]
df_top_cities_top_10

In [ ]:
import plotly.express as px
fig = px.scatter_mapbox(
    df_top_cities_top_10, 
    lat="lat", 
    lon="lon",
    color="total_score",
    size="total_score", 
    color_continuous_scale=px.colors.cyclical.IceFire,
    size_max=15, # Taille maximale des bulles pour que ça reste lisible
    zoom=4, 
    center={"lat": 46.2276, "lon": 2.2137}, # Centré sur la France
    mapbox_style="carto-positron", 
    hover_name="city",
    title="Top 10 Destinations en France"
)

fig.show()

Pour les besoins de l'exercice et s'assurer d'avoir un volume de données suffisant, nous avons simulé une recherche d'hôtel à M+1, indépendamment de la météo à J+5.

In [ ]:
!cd booking_scraper_project && scrapy crawl booking_spider -O ../hotels.json

In [ ]:
df_hotels = pd.read_json("hotels.json")

In [ ]:
df_hotels_without_na = df_hotels.dropna()
df_hotels_without_na.head()

In [ ]:
df_top_cities['id'] = df_top_cities.apply(lambda x: uuid.uuid4(), axis=1)
print(df_top_cities.head())
df_top_cities.to_csv("final_output/villes_table.csv", index=False)

In [ ]:
print(df_weather_groupby.head())
print(df_weather_groupby.shape)
print(df_hotels_without_na.shape)

In [ ]:
df_top_cities_light = df_top_cities[['id', 'city']]
# 1. Reset l'index pour sortir 'city' et 'date'
df_weather_flat = df_weather_groupby.reset_index()

# 2. Aplatir les noms de colonnes (si MultiIndex)
# On prend le niveau 0 (ex: 'temp') sauf si c'est vide, sinon on garde le niveau 1
#df_weather_flat.columns = ['_'.join(c).strip('_') for c in df_weather_flat.columns.to_flat_index()]
df_weather_flat.columns = ['_'.join(c).strip('_') for c in df_weather_flat.columns.to_flat_index()]

# Vérifie que les colonnes sont bien 'city', 'date', 'temp_max', etc.
print(df_weather_flat.columns)

df_weather_flat_light = df_weather_flat[['city', 'date', 'temp_mean', 'temp_min',
       'temp_max', 'prob_rain_max', 'volume_rain_mean', 'volume_rain_max',
       'volume_rain_sum', 'wind_speed_max', 'perc_cloud_mean', 'score_temp',
       'score_rain_prob', 'score_rain_vol', 'score_wind', 'score_cloud',
       'total_score']]

# 3. Maintenant le merge
df_weather_final = df_weather_flat_light.merge(df_top_cities_light, on='city')
df_weather_final = df_weather_final.drop(['city'], axis=1)
print(df_weather_final.head())

df_weather_final.to_csv("final_output/weather_table.csv", index=False)

In [ ]:
df_top_cities_light = df_top_cities[['id', 'city']]

# 3. Maintenant le merge
df_hotel_final = df_hotels_without_na.merge(df_top_cities_light, on='city')
df_hotel_final = df_hotel_final.drop(['city'], axis=1)
print(df_hotel_final.head())

df_hotel_final.to_csv("final_output/hotels_table.csv", index=False)